# Lesson 03 - Agentic Design Patterns

V této lekci prozkoumáme tři základní návrhové vzory pro tvorbu efektivních AI agentů:

1. **Jasné instrukce pro agenta** — Vytváření přesných, rolí definujících požadavků, které řídí chování agenta  
2. **Strukturovaný výstup s Pydantic modely** — Zajištění, že agenti vracejí předvídatelná a validovaná data  
3. **Agenti s jednoznačnou odpovědností** — Návrh zaměřených agentů, kteří každý zvládnou jednu věc dobře

Každý vzor použijeme na scénář **doporučovače cestovních destinací**, postupně vytvářejícího systém, který dokáže navrhnout destinace, zkontrolovat dostupnost a řídit logistiku.


## Nastavení


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Vzor 1: Jasné pokyny pro agenta

Nejsilnějším vzorem je také ten nejjednodušší: psaní jasných, podrobných pokynů pro vašeho agenta.

Dobré pokyny definují:
- **Kdo** je agent (persona a tón)
- **Co** má dělat (krok za krokem odpovědnosti)
- **Jak** se má chovat (omezení a styl)

Níže vytváříme cestovního concierge agenta s explicitními pokyny, které ovlivňují každou odpověď, kterou vyprodukuje.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Vzor 2: Strukturovaný výstup pomocí Pydantic modelů

Volný text je užitečný pro konverzaci, ale downstream systémy potřebují strukturovaná data.  
Spárováním **Pydantic modelů** s **nástrojovou funkcí** můžeme:

- Definovat přesné schéma pro výstup agenta  
- Automaticky ověřovat odpovědi  
- Spolehlivě integrovat výsledky agenta do aplikační logiky  

Také představujeme nástroj, který vrací podrobnosti o cílové destinaci, aby agent mohl zakládat svá doporučení na reálných datech.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Vzor 3: Agenti s jednou odpovědností

Složitým úkolům prospívá rozdělení práce mezi více specializovaných agentů, z nichž každý má jedinou odpovědnost:

- **Odborník na destinace**, který zná místa a dostupnost
- **Plánovač logistiky**, který řeší lety, hotely a itineráře

Toto odpovídá zásadě softwarového inženýrství *oddělení zodpovědností* — každý agent je jednodušší testovat, udržovat a nezávisle vylepšovat.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Shrnutí

V této lekci jsme použili tři vzory agentního designu na scénář doporučování cestování:

| Vzor | Klíčová myšlenka | Výhoda |
|---|---|---|
| **Jasné pokyny** | Na začátku definujte personu, odpovědnosti a omezení | Konzistentní, značkově odpovídající chování agenta |
| **Strukturovaný výstup** | Používejte Pydantic modely jako formát odpovědi | Validované, strojově čitelné výsledky |
| **Jedna odpovědnost** | Každému agentovi přiřaďte jedno zaměřené úkol | Snazší testování, údržba a skládání |

Tyto vzory se přirozeně doplňují — můžete kombinovat jasné pokyny se strukturovaným výstupem uvnitř agenta s jednou odpovědností, abyste vytvořili robustní, produkčně připravené systémy.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o vyloučení odpovědnosti**:  
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o přesnost, mějte, prosím, na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Původní dokument v jeho mateřském jazyce by měl být považován za závazný zdroj. Pro kritické informace je doporučen profesionální lidský překlad. Nejsme odpovědní za jakákoliv nedorozumění nebo chybné interpretace vyplývající z použití tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
